# Odoo Business Onboarding Flow — XML-RPC API

End-to-end automated provisioning of a new company tenant in Odoo 17+ via the XML-RPC API.

## Flow Overview

| Phase | What Gets Provisioned |
|-------|-----------------------|
| **1. Company** | `res.company` root tenant record |
| **2. Admin User** | `res.users` with full ERP + Accounting roles |
| **3. CoA & Product** | Chart of Accounts entries + a global service product |
| **4. Journals** | Sales journal (invoicing) + Bank journal (payments) |
| **5. Order Flow** | Guest partner → Invoice → Bank payment |

> **Note:** `TARGET_COMPANY_CONTEXT` is passed on every multi-company-sensitive write to satisfy
> Odoo's internal access checks and prevent company-isolation validation errors.


## Setup — Server Connection & Authentication

Establishes two XML-RPC endpoints:
- `/xmlrpc/2/common` — unauthenticated (used only for `authenticate`)
- `/xmlrpc/2/object` — authenticated (all model operations go here)


In [19]:
import xmlrpc.client

# ── Server configuration ──────────────────────────────────────────────────────
URL      = "http://localhost:8069"
DB       = "m.nushath"
USERNAME = "user@company.com"
API_KEY  = "user"
# ─────────────────────────────────────────────────────────────────────────────

common = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/common', allow_none=True)
models = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/object', allow_none=True)

uid = common.authenticate(DB, USERNAME, API_KEY, {})
assert uid, "[CRITICAL] Authentication failed — check DB, USERNAME, and API_KEY."

print(f"[OK] Authenticated | UID: {uid} | DB: {DB} | Host: {URL}")

[OK] Authenticated | UID: 5 | DB: m.nushath | Host: http://localhost:8069


---
## Phase 1 — Company Onboarding

Creates the root `res.company` record. Odoo automatically provisions a linked `res.partner`
(the company's public contact) and assigns a default currency.

The returned `company_id` is promoted to `TARGET_COMPANY_ID` in the next cell and used
as the anchor for every subsequent resource created in this flow.


In [20]:
new_company_vals = {
    'name':   'Soconi Global Trading Ltd - Test 2',
    'email':  'operations@soconi.global',
    'phone':  '+1-800-555-0155',
    'street': '100 Innovation Way',
    'city':   'Tech City',
    'zip':    '90210',
}

# Passing a single dict inside a list returns a raw integer ID
company_id = models.execute_kw(DB, uid, API_KEY, 'res.company', 'create', [new_company_vals])
print(f"[OK] Company created | ID: {company_id}")

# Read back auto-provisioned metadata to confirm the record
company_meta = models.execute_kw(
    DB, uid, API_KEY, 'res.company', 'read',
    [[company_id]], {'fields': ['name', 'partner_id', 'currency_id']}
)[0]

print(f"     Name:          {company_meta['name']}")
print(f"     Partner ID:    {company_meta['partner_id'][0]} ({company_meta['partner_id'][1]})")
print(f"     Base Currency: {company_meta['currency_id'][1]}")

[OK] Company created | ID: 6
     Name:          Soconi Global Trading Ltd - Test 2
     Partner ID:    48 (Soconi Global Trading Ltd - Test 2)
     Base Currency: USD


In [21]:
# Promote to canonical constant used throughout the rest of this flow.
TARGET_COMPANY_ID = company_id

# Passed on every multi-company-sensitive write to satisfy Odoo's access checks.
TARGET_COMPANY_CONTEXT = {
    'company_id':          TARGET_COMPANY_ID,
    'allowed_company_ids': [TARGET_COMPANY_ID],
}

print(f"[OK] TARGET_COMPANY_ID      = {TARGET_COMPANY_ID}")
print(f"[OK] TARGET_COMPANY_CONTEXT = {TARGET_COMPANY_CONTEXT}")

[OK] TARGET_COMPANY_ID      = 6
[OK] TARGET_COMPANY_CONTEXT = {'company_id': 6, 'allowed_company_ids': [6]}


---
## Phase 2 — Admin User Provisioning

Creates a company-scoped internal user with full administrative rights.

### Permission groups assigned

| XML ID | Module | Effect |
|--------|--------|--------|
| `group_erp_manager` | `base` | ERP Administration access |
| `group_system` | `base` | Sets the **Administrator** radio button |
| `group_sale_salesman` | `sales_team` | Sales access |
| `group_account_manager` | `account` | Accounting Manager access |

**Schema discovery** is run first to dynamically resolve the Many2many field name that links
`res.users` → `res.groups`. This avoids hardcoding `groups_id` vs `group_ids` across Odoo versions.


In [22]:
# Dynamically resolve the Many2many field linking res.users → res.groups.
# Version-safe: the field was renamed between Odoo versions.
user_schema = models.execute_kw(DB, uid, API_KEY, 'res.users', 'fields_get',
    [], {'attributes': ['type', 'relation']}
)

dynamic_group_field = next(
    (f for f, p in user_schema.items()
     if p.get('type') == 'many2many' and p.get('relation') == 'res.groups'),
    None
)
assert dynamic_group_field, "[CRITICAL] No user → group relation field found in schema."
print(f"[OK] Group field: '{dynamic_group_field}'")

# Resolve stable database IDs from XML IDs — robust across database reinstalls.
required_permissions = {
    'admin_access': ('base',       'group_erp_manager'),
    'admin_system': ('base',       'group_system'),        # Sets the "Administrator" radio
    'sales':        ('sales_team', 'group_sale_salesman'),
    'accounting':   ('account',    'group_account_manager'),
}

group_db_ids = []
for key, (module, xml_name) in required_permissions.items():
    meta = models.execute_kw(DB, uid, API_KEY, 'ir.model.data', 'search_read', [
        [('module', '=', module), ('name', '=', xml_name)]
    ], {'fields': ['res_id'], 'limit': 1})
    if meta:
        group_db_ids.append(meta[0]['res_id'])

print(f"[OK] Resolved group IDs: {group_db_ids}")

[OK] Group field: 'group_ids'
[OK] Resolved group IDs: [2, 4, 28, 36]


In [ ]:
NEW_LOGIN    = 'janeD.admin@soconi.global'
NEW_PASSWORD = 'SecurePassword123!'

user_payload = {
    'name':     'Jane Doe (Executive Administrator)',
    'login':    NEW_LOGIN,
    'email':    'janeD.admin@soconi.global',
    'password': NEW_PASSWORD,
    'company_id':  TARGET_COMPANY_ID,
    'company_ids': [(6, 0, [TARGET_COMPANY_ID])],
    'share':    False,
    # (6, 0, [...]) replaces the default group list with our resolved IDs atomically
    dynamic_group_field: [(6, 0, group_db_ids)],
}

# Context is required here to pass Odoo's company-isolation checks during creation
new_user_id = models.execute_kw(
    DB, uid, API_KEY, 'res.users', 'create',
    [user_payload], {'context': TARGET_COMPANY_CONTEXT}
)
print(f"[OK] Admin user created | ID: {new_user_id} | Login: {NEW_LOGIN}")

[OK] Admin user created | ID: 28 | Login: janeD.admin@soconi.global


In [25]:
# Verify the new user can authenticate independently before proceeding
uid_new = common.authenticate(DB, NEW_LOGIN, NEW_PASSWORD, {})
assert uid_new, "[CRITICAL] New user authentication failed."
print(f"[OK] Authenticated as new user | UID: {uid_new}")

[OK] Authenticated as new user | UID: 28


---
## Phase 3 — Chart of Accounts & Global Product

Sets up the minimal accounting structure required to post invoices.

### Accounts created

| Code | Name | Type | Notes |
|------|------|------|-------|
| `120000` | Accounts Receivable | `asset_receivable` | `reconcile: True` required to match payments to invoices |
| `400000` | Product Sales | `income` | Revenue account; default on the Sales journal |
| `101400` | Main Bank Account | `asset_cash` | Cleared cash balance |
| `101401` | Outstanding Receipts | `asset_current` | Payments registered but not yet bank-confirmed |
| `101402` | Bank Suspense | `asset_current` | Unmatched bank statement lines; auto-cleared on match |

> **Odoo 17+ note:** `account.account` uses plural `company_ids` (Many2many). Using singular
> `company_id` will silently fail to scope the account to a company.

The **global dummy product** (`company_id: False`) is a zero-price service product used as the
line item on all programmatic invoices. Setting `company_id` to `False` makes it visible across
every company in the instance — no per-company duplication needed.


In [27]:
product_payload = {
    'name':           'Dummy Product (Global Scope)',
    'default_code':   'DUMMY-GLOBAL',
    'type':           'service',    # No inventory tracking
    'list_price':     0.0,
    'standard_price': 0.0,
    'sale_ok':        True,
    'purchase_ok':    True,
    'company_id':     False,         # False = shared across all companies
}

DUMMY_PRODUCT_ID = models.execute_kw(
    DB, uid, API_KEY, 'product.template', 'create',
    [product_payload], {'context': TARGET_COMPANY_CONTEXT}
)
print(f"[OK] Global dummy product created | ID: {DUMMY_PRODUCT_ID}")

# Verify global scope — Odoo returns False for unset Many2one fields
product_check = models.execute_kw(
    DB, uid, API_KEY, 'product.template', 'read',
    [[DUMMY_PRODUCT_ID]], {'fields': ['name', 'type', 'company_id']}
)[0]

scope = "Global (all companies)" if not product_check['company_id'] else f"Restricted to company {product_check['company_id'][0]}"
print(f"     Name:  {product_check['name']}")
print(f"     Type:  {product_check['type'].upper()}")
print(f"     Scope: {scope}")

[OK] Global dummy product created | ID: 17
     Name:  Dummy Product (Global Scope)
     Type:  SERVICE
     Scope: Global (all companies)


In [28]:
# Odoo 17+: account.account ownership uses plural company_ids (Many2many), not singular company_id
ar_payload = {
    'name':         'Accounts Receivable - PC1',
    'code':         '120000',
    'account_type': 'asset_receivable',
    'reconcile':    True,    # Required — enables matching payments to invoices
    'company_ids':  [(6, 0, [TARGET_COMPANY_ID])],
}

sales_payload = {
    'name':         'Product Sales - PC1',
    'code':         '400000',
    'account_type': 'income',
    'reconcile':    False,
    'company_ids':  [(6, 0, [TARGET_COMPANY_ID])],
}

AR_ACCOUNT_ID    = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'account.account', 'create', [ar_payload])
SALES_ACCOUNT_ID = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'account.account', 'create', [sales_payload])

print(f"[OK] Accounts Receivable | ID: {AR_ACCOUNT_ID}    (code: 120000)")
print(f"[OK] Sales Income        | ID: {SALES_ACCOUNT_ID}    (code: 400000)")

[OK] Accounts Receivable | ID: 127    (code: 120000)
[OK] Sales Income        | ID: 128    (code: 400000)


In [29]:
# Three accounts power the standard Odoo bank reconciliation flow:
#   Main Bank          — cleared/confirmed cash balance
#   Outstanding Receipts — payments registered but not yet bank-statement-confirmed
#   Bank Suspense      — unmatched statement lines; auto-cleared on reconciliation
main_bank_payload = {
    'name':         'Main Bank Account - PC1',
    'code':         '101400',
    'account_type': 'asset_cash',    # Liquid cash
    'reconcile':    False,            # Asset accounts do not reconcile line-by-line
    'company_ids':  [(6, 0, [TARGET_COMPANY_ID])],
}

outstanding_receipts_payload = {
    'name':         'Outstanding Receipts - PC1',
    'code':         '101401',
    'account_type': 'asset_current',
    'reconcile':    True,             # Must be True — matched against bank statements
    'company_ids':  [(6, 0, [TARGET_COMPANY_ID])],
}

suspense_payload = {
    'name':         'Bank Suspense Account - PC1',
    'code':         '101402',
    'account_type': 'asset_current',
    'reconcile':    True,             # Auto-cleared by Odoo's reconciliation widget
    'company_ids':  [(6, 0, [TARGET_COMPANY_ID])],
}

MAIN_BANK_ACCOUNT_ID           = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'account.account', 'create', [main_bank_payload])
OUTSTANDING_RECEIPTS_ACCOUNT_ID = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'account.account', 'create', [outstanding_receipts_payload])
SUSPENSE_ACCOUNT_ID            = models.execute_kw(DB, uid_new, NEW_PASSWORD, 'account.account', 'create', [suspense_payload])

print(f"[OK] Main Bank Account    | ID: {MAIN_BANK_ACCOUNT_ID}   (code: 101400)")
print(f"[OK] Outstanding Receipts | ID: {OUTSTANDING_RECEIPTS_ACCOUNT_ID}   (code: 101401)")
print(f"[OK] Bank Suspense        | ID: {SUSPENSE_ACCOUNT_ID}   (code: 101402)")

[OK] Main Bank Account    | ID: 129   (code: 101400)
[OK] Outstanding Receipts | ID: 130   (code: 101401)
[OK] Bank Suspense        | ID: 131   (code: 101402)


---
## Phase 4 — Journal Configuration

Journals are named ledgers that group related accounting entries.

| Journal | Code | Type | Purpose |
|---------|------|------|---------|
| Sales | `INV` | `sale` | Customer invoices — auto-assigned by Odoo on `out_invoice` creation |
| Bank | `BNK` | `bank` | Cash received / payments processed |


In [30]:
sales_journal_payload = {
    'name':               'Sales',
    'code':               'INV',     # Prefix for invoice sequences: INV/2026/0001
    'type':               'sale',    # Marks this as the Customer Invoices journal
    'company_id':         TARGET_COMPANY_ID,
    'default_account_id': SALES_ACCOUNT_ID,   # Revenue posts to account 400000
}

SALES_JOURNAL_ID = models.execute_kw(
    DB, uid_new, NEW_PASSWORD, 'account.journal', 'create', [sales_journal_payload]
)
print(f"[OK] Sales Journal created | ID: {SALES_JOURNAL_ID}")

# Verify
journal_check = models.execute_kw(
    DB, uid_new, NEW_PASSWORD, 'account.journal', 'read',
    [[SALES_JOURNAL_ID]], {'fields': ['name', 'code', 'type', 'default_account_id']}
)[0]
linked = journal_check['default_account_id'][1] if journal_check['default_account_id'] else "None"
print(f"     Journal:         {journal_check['name']} ({journal_check['code']})")
print(f"     Type:            {journal_check['type'].upper()}")
print(f"     Default Account: {linked}")

[OK] Sales Journal created | ID: 45
     Journal:         Sales (INV)
     Type:            SALE
     Default Account: 400000 Product Sales - PC1


### Bank journal — 3-step payment wiring

Creating a `bank` journal auto-generates inbound payment method lines. These lines are immediately
patched to route incoming payments through **Outstanding Receipts** before final bank reconciliation:

```
Register Payment → Outstanding Receipts → Bank Statement Match → Main Bank Account
```

In [31]:
bank_journal_payload = {
    'name':                'Bank',
    'code':                'BNK',
    'type':                'bank',
    'company_id':          TARGET_COMPANY_ID,
    'default_account_id':  MAIN_BANK_ACCOUNT_ID,
    'suspense_account_id': SUSPENSE_ACCOUNT_ID,
}

bank_journal_id = models.execute_kw(
    DB, uid, API_KEY, 'account.journal', 'create', [bank_journal_payload]
)
print(f"[OK] Bank Journal created | ID: {bank_journal_id}")

# Odoo auto-generates inbound payment method lines on journal creation.
# Wire them to Outstanding Receipts to enable the 3-step reconciliation flow.
inbound_lines = models.execute_kw(
    DB, uid, API_KEY, 'account.payment.method.line', 'search_read',
    [[('journal_id', '=', bank_journal_id), ('payment_type', '=', 'inbound')]],
    {'fields': ['id', 'name']}
)

if inbound_lines:
    line_ids = [line['id'] for line in inbound_lines]
    models.execute_kw(DB, uid, API_KEY, 'account.payment.method.line', 'write', [
        line_ids, {'payment_account_id': OUTSTANDING_RECEIPTS_ACCOUNT_ID}
    ])
    print(f"[OK] Inbound payment lines {line_ids} → wired to Outstanding Receipts")
else:
    print("[WARN] No inbound payment lines found — skipping wire step.")

# Verify full journal architecture
journal_data = models.execute_kw(
    DB, uid, API_KEY, 'account.journal', 'read',
    [[bank_journal_id]], {'fields': ['name', 'default_account_id', 'suspense_account_id']}
)[0]
print(f"\n     Journal:          {journal_data['name']}")
print(f"     Main Account:     {journal_data['default_account_id'][1] if journal_data['default_account_id'] else 'None'}")
print(f"     Suspense Account: {journal_data.get('suspense_account_id')[1] if journal_data.get('suspense_account_id') else 'None'}")

if inbound_lines:
    verify_lines = models.execute_kw(
        DB, uid, API_KEY, 'account.payment.method.line', 'read',
        [line_ids], {'fields': ['name', 'payment_account_id']}
    )
    for line in verify_lines:
        acc = line.get('payment_account_id')
        print(f"     Inbound '{line['name']}' → {acc[1] if acc else 'UNASSIGNED'}")

[OK] Bank Journal created | ID: 46
[OK] Inbound payment lines [51] → wired to Outstanding Receipts

     Journal:          Bank
     Main Account:     Main Bank Account - PC1
     Suspense Account: Bank Suspense Account - PC1
     Inbound 'Manual Payment' → Outstanding Receipts - PC1


---
## Phase 5 — Guest Checkout & Invoice Flow

Simulates an e-commerce order: a guest customer places an order, an invoice is raised,
and payment is registered immediately against the Bank journal.

### Steps
1. **Guest partner** — a `res.partner` scoped to the company (archived after creation)
2. **Bank Journal** — fetched dynamically by type; decoupled from the creation step above
3. **Invoice** — an `out_invoice` referencing the dummy product and an external order ID
4. **Payment** — the `account.payment.register` wizard settles the invoice in one API call

The external order reference (`WEB-ORD-*`) populates both `ref` (journal reference column)
and `payment_reference` (bank statement memo) for clean reconciliation.


In [32]:
# Simulated incoming order payload from an external system (website / e-commerce platform)
incoming_order_data = {
    'name':     'Alex Guest',
    'email':    'alex.guest@example.com',
    'amount':   199.99,
    'quantity': 1.0,
}

partner_payload = {
    'name':       incoming_order_data['name'],
    'email':      incoming_order_data['email'],
    'active':     False,   # Archived immediately — guest checkout pattern
    'company_id': TARGET_COMPANY_ID,
    'comment':    'Auto-generated Guest Checkout',
}

CUSTOMER_PARTNER_ID = models.execute_kw(
    DB, uid_new, NEW_PASSWORD, 'res.partner', 'create', [partner_payload]
)
print(f"[OK] Guest partner created | ID: {CUSTOMER_PARTNER_ID} | Name: {incoming_order_data['name']}")

[OK] Guest partner created | ID: 51 | Name: Alex Guest


In [33]:
# Fetch by type rather than hardcoding the journal ID — resilient to re-runs
bank_journals = models.execute_kw(
    DB, uid, API_KEY, 'account.journal', 'search_read',
    [[('type', '=', 'bank'), ('company_id', '=', TARGET_COMPANY_ID)]],
    {'fields': ['id', 'name'], 'limit': 1}
)
assert bank_journals, "[CRITICAL] No Bank journal found for this company."

BANK_JOURNAL_ID = bank_journals[0]['id']
print(f"[OK] Bank Journal located | ID: {BANK_JOURNAL_ID} | Name: {bank_journals[0]['name']}")

[OK] Bank Journal located | ID: 46 | Name: Bank


In [34]:
from datetime import date

external_order_id = "WEB-ORD-88219"

invoice_payload = {
    'move_type':         'out_invoice',
    'partner_id':        CUSTOMER_PARTNER_ID,
    'company_id':        TARGET_COMPANY_ID,
    'ref':               external_order_id,       # Populates the Reference column on the journal entry
    'payment_reference': external_order_id,       # Used as the bank payment communication memo
    'invoice_line_ids':  [(0, 0, {
        'product_id': DUMMY_PRODUCT_ID,
        'quantity':   1.0,
        'price_unit': incoming_order_data['amount'],
        'name':       f'Checkout Payment for {external_order_id}',
    })],
}

invoice_id = models.execute_kw(
    DB, uid, API_KEY, 'account.move', 'create',
    [invoice_payload], {'context': TARGET_COMPANY_CONTEXT}
)
# action_post confirms the draft and generates the immutable journal entry
models.execute_kw(
    DB, uid, API_KEY, 'account.move', 'action_post',
    [[invoice_id]], {'context': TARGET_COMPANY_CONTEXT}
)
print(f"[OK] Invoice created and posted | ID: {invoice_id} | Ref: {external_order_id}")

# account.payment.register is a transient wizard — create then trigger in two calls
wizard_context = {
    'company_id':          TARGET_COMPANY_ID,
    'allowed_company_ids': [TARGET_COMPANY_ID],
    'active_model':        'account.move',
    'active_ids':          [invoice_id],
}

payment_wizard_payload = {
    'journal_id':   BANK_JOURNAL_ID,
    'amount':       incoming_order_data['amount'],
    'payment_date': str(date.today()),
}

wizard_id = models.execute_kw(
    DB, uid, API_KEY, 'account.payment.register', 'create',
    [payment_wizard_payload], {'context': wizard_context}
)
models.execute_kw(
    DB, uid, API_KEY, 'account.payment.register', 'action_create_payments',
    [[wizard_id]], {'context': wizard_context}
)
print(f"[OK] Payment registered | Invoice {invoice_id} fully settled via Bank Journal {BANK_JOURNAL_ID}")

[OK] Invoice created and posted | ID: 52 | Ref: WEB-ORD-88219
[OK] Payment registered | Invoice 52 fully settled via Bank Journal 46
